## Pre-Built Invoice Processing with Azure AI Content Understanding

### Installing Required Libraries

In [ ]:
%pip install python-dotenv azure-ai-contentunderstanding

### Setting up the Environment

Create a `.env` file with the endpoint and key for an Azure AI Content Understanding resource. The endpoint should look like `https://<resource-name>.services.ai.azure.com/`.

```text
FOUNDRY_AI_SERVICES_ENDPOINT=https://<resource-name>.services.ai.azure.com/
FOUNDRY_AI_SERVICES_KEY=<your-resource-key>
```

Before using `prebuilt-invoice`, deploy `gpt-4.1` and `text-embedding-3-large` in the Foundry resource and configure the default model deployments once. See Microsoft's [model deployment setup sample](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-contentunderstanding_1.1.0/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_update_defaults.py).

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput, AnalysisResult

load_dotenv()

endpoint = os.getenv("FOUNDRY_AI_SERVICES_ENDPOINT")
api_version = "2026-06-01-preview"


### Creating the Content Understanding Client

In [3]:
from azure.identity import DefaultAzureCredential

# Authenticate with Entra ID (run `az login` first)
credential = DefaultAzureCredential()

client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential= credential,
    api_version=api_version,
)

### Analyzing the Local Sample Invoice

In [4]:
sample_path = Path("Samples") / "sample-invoice.pdf"
if not sample_path.is_file():
    sample_path = (
        Path("Information-Extraction")
        / "Document-Intelligence"
        / "Pre-Built-Offerings"
        / "Samples"
        / "sample-invoice.pdf"
    )

if not sample_path.is_file():
    raise FileNotFoundError(
        "Could not find Samples/sample-invoice.pdf. Run this notebook from its folder or the repository root."
    )

sample_bytes = sample_path.read_bytes()

In [34]:
poller = client.begin_analyze(
    analyzer_id="prebuilt-invoice",
    inputs=[
        AnalysisInput(
            data=sample_bytes,
            name=sample_path.name,
            mime_type="application/pdf",
        )
    ],
)
result: AnalysisResult = poller.result()

if not result.contents:
    raise ValueError("The invoice analysis returned no document content.")

invoice = result.contents[0]
fields = invoice.fields or {}

customer_name = fields.get("CustomerName")
invoice_date = fields.get("InvoiceDate")
print("Customer name:", customer_name.value if customer_name else "N/A")
print("Invoice date:", invoice_date.value if invoice_date else "N/A")

invoice_total = fields.get("TotalAmount")
amount = invoice_total.value.get("Amount").get("valueNumber")
currency = invoice_total.value.get("CurrencyCode").get("valueString")
print(f"Total: {amount} {currency}")

Customer name: MICROSOFT CORPORATION
Invoice date: 2019-11-15
Total: 610 USD


In [35]:
import pprint
pprint.pp(fields)

{'AmountDue': {'type': 'object', 'valueObject': {'Amount': {'type': 'number', 'valueNumber': 610, 'spans': [{'offset': 1227, 'length': 7}], 'confidence': 0.819, 'source': 'D(1,7.3586,7.8037,7.9324,7.8075,7.9314,7.9690,7.3576,7.9653)'}, 'CurrencyCode': {'type': 'string', 'valueString': 'USD'}}},
 'BalanceForward': {'type': 'object', 'valueObject': {'Amount': {'type': 'number', 'valueNumber': 500, 'spans': [{'offset': 1180, 'length': 7}], 'confidence': 0.796, 'source': 'D(1,7.3628,7.5080,7.9280,7.5107,7.9272,7.6733,7.3620,7.6706)'}, 'CurrencyCode': {'type': 'string', 'valueString': 'USD'}}},
 'BillingAddress': {'type': 'string', 'valueString': '123 Bill St, Redmond WA, 98052', 'spans': [{'offset': 374, 'length': 12}, {'offset': 387, 'length': 17}], 'confidence': 0.643, 'source': 'D(1,0.5785,4.3554,1.2870,4.3571,1.2866,4.5146,0.5781,4.5129);D(1,0.5762,4.5576,2.0285,4.5557,2.0287,4.7185,0.5764,4.7204)'},
 'BillingAddressRecipient': {'type': 'string', 'valueString': 'Microsoft Finance', 'sp

In [38]:
# visualize content markdown
from IPython.display import Markdown, display
display(Markdown(result.contents[0].markdown))

CONTOSO LTD.


# INVOICE

Contoso Headquarters
123 456th St
New York, NY, 10001


<table>
<tr>
<td>INVOICE: INV-100</td>
</tr>
<tr>
<th>DATE: 11/15/2019</th>
</tr>
<tr>
<th>DUE DATE: 12/15/2019</th>
</tr>
<tr>
<th>CUSTOMER NAME: MICROSOFT CORPORATION<br>CUSTOMER ID: CID-12345</th>
</tr>
</table>


Microsoft Corp
123 Other St,
Redmond WA, 98052

BILL TO:
Microsoft Finance
123 Bill St,
Redmond WA, 98052

SHIP TO:
Microsoft Delivery
123 Ship St,
Redmond WA, 98052

SERVICE ADDRESS:
Microsoft Services
123 Service St,
Redmond WA, 98052


<table>
<tr>
<th>SALESPERSON</th>
<th>P.O. NUMBER</th>
<th>REQUISITIONER</th>
<th>SHIPPED VIA</th>
<th>F.O.B. POINT</th>
<th>TERMS</th>
</tr>
<tr>
<td></td>
<td>PO-3333</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<th>QUANTITY</th>
<th>DESCRIPTION</th>
<th>UNIT PRICE</th>
<th>TOTAL</th>
</tr>
<tr>
<td>1</td>
<td>Test for 23 fields</td>
<td>1</td>
<td>$100.00</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
</table>


<table>
<tr>
<td>SUBTOTAL</td>
<td>$100.00</td>
</tr>
<tr>
<td>SALES TAX</td>
<td>$10.00</td>
</tr>
<tr>
<td>TOTAL</td>
<td>$110.00</td>
</tr>
<tr>
<td>PREVIOUS BALANCE</td>
<td>$500.00</td>
</tr>
<tr>
<td>TOTAL DUE</td>
<td>$610.00</td>
</tr>
</table>


THANK YOU FOR YOUR BUSINESS!

REMIT TO:
Contoso Billing
123 Remit St
New York, NY, 10001
